# 01 · Layer 1：Agent 的三個身分欄位

上一章我們讓 agent 動起來了。這一章拆開來看：一個 `LlmAgent` 到底由什麼構成？

```
             ┌──────────── LlmAgent ────────────┐
             │  name         我是誰（給系統看）  │
             │  description  我會什麼（給別的 agent 看）│
             │  instruction  我怎麼做（給模型看）│
             │  model        我用哪個腦          │
             │  tools        我的手腳            │
             └──────────────────────────────────┘
```

前三個是「身分」，本章的主題。`model` 在第 03 章，`tools` 在第 02 章。

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. `name`：不只是變數名稱

`name` 是 agent 在**事件串流**裡的署名。單一 agent 時你可能覺得無所謂，
但一旦有多個 agent，`event.author` 就是你唯一能分辨「現在是誰在講話」的依據。

規則：只能用英文、數字、底線，而且在同一個系統裡不能重複。

In [2]:
from google.adk.agents import LlmAgent

agent = LlmAgent(
    name="tutor",
    model=get_model(),
    instruction="你是 Python 教學助理，用繁體中文回答。",
)
print("name =", agent.name)

try:
    LlmAgent(name="my agent", model=get_model(), instruction="x")
except Exception as exc:
    print(f"\n名稱含空白會被擋下來：\n  {type(exc).__name__}: {str(exc)[:160]}")

name = tutor

名稱含空白會被擋下來：
  ValidationError: 1 validation error for LlmAgent
name
  Value error, Node name 'my agent' must be a valid Python identifier. [type=value_error, input_value='my agent', input_typ


## 2. `instruction`：模型真正讀到的東西

`instruction` 就是系統提示。它決定行為，而且**寫得含糊模型就自由發揮**。

底下用同一個問題問兩個 agent，差別只在 instruction 的精確度。

In [3]:
vague = LlmAgent(
    name="vague_bot",
    model=get_model(),
    instruction="你會回答問題。",
)

precise = LlmAgent(
    name="precise_bot",
    model=get_model(),
    instruction=(
        "你是資深 Python 工程師，負責回答初學者的問題。\n"
        "規則：\n"
        "1. 一律用繁體中文。\n"
        "2. 先給一句話的結論，再給最多三點說明。\n"
        "3. 一定要附一段可執行的程式碼範例。\n"
        "4. 不要說「這是一個好問題」之類的開場白。"
    ),
)

question = "Python 的 list 和 tuple 差在哪？"

print("=" * 60)
print("【含糊版】")
print(await run_once(vague, question))
print("=" * 60)
print("【精確版】")
print(await run_once(precise, question))

【含糊版】


Python 的 `list` 和 `tuple` 都是用來儲存多個項目的序列，但它們之間有幾個關鍵的差異：

**1. 可變性 (Mutability)**

*   **`list` 是可變的 (Mutable):** 這意味著你可以修改 `list` 的內容，包括新增、刪除或更改其中的元素。
    ```python
    my_list = [1, 2, 3]
    my_list.append(4)      # 新增元素
    my_list[0] = 10        # 修改元素
    del my_list[1]         # 刪除元素
    print(my_list)         # 輸出: [10, 3, 4]
    ```

*   **`tuple` 是不可變的 (Immutable):** 這意味著一旦創建了 `tuple`，就無法再修改它的內容。你無法新增、刪除或更改其中的元素。
    ```python
    my_tuple = (1, 2, 3)
    # my_tuple.append(4)   # 這會引發 AttributeError
    # my_tuple[0] = 10     # 這會引發 TypeError
    # del my_tuple[1]      # 這會引發 TypeError
    print(my_tuple)        # 輸出: (1, 2, 3)
    ```
    如果你需要修改 `tuple` 的內容，你需要創建一個新的 `tuple`。

**2. 語法 (Syntax)**

*   **`list` 使用方括號 `[]`:**
    ```python
    my_list = [1, "hello", 3.14]
    ```

*   **`tuple` 使用圓括號 `()`:**
    ```python
    my_tuple = (1, "hello", 3.14)
    ```
    **注意:**
    *   創建單一元素的 `tuple` 時，需要在元素後面加上一個逗號，否則它會被解釋為該元素的類型。
        ```python
        single_element_tuple = 

Python 的 list 和 tuple 主要差異在於它們的**可變性**。

*   **List 是可變的 (mutable)**：你可以隨時修改 list 的內容，例如新增、刪除或更改元素。
*   **Tuple 是不可變的 (immutable)**：一旦創建，tuple 的內容就不能被修改。
*   **語法不同**：List 使用方括號 `[]`，而 tuple 使用圓括號 `()`。

```python
# List 的範例 (可變)
my_list = [1, 2, 3]
print(f"原始 list: {my_list}")
my_list.append(4)  # 新增元素
print(f"新增元素後: {my_list}")
my_list[0] = 10    # 修改元素
print(f"修改元素後: {my_list}")

# Tuple 的範例 (不可變)
my_tuple = (1, 2, 3)
print(f"\n原始 tuple: {my_tuple}")
# 嘗試修改 tuple 會發生錯誤：
# my_tuple.append(4) # 這會引發 AttributeError
# my_tuple[0] = 10   # 這會引發 TypeError

# 雖然 tuple 本身不可變，但如果 tuple 包含可變物件 (如 list)，則該物件內的元素可以被修改
my_complex_tuple = ([1, 2], 3)
print(f"\n包含 list 的 tuple: {my_complex_tuple}")
my_complex_tuple[0].append(4) # 可以修改 list 內的元素
print(f"修改 list 元素後: {my_complex_tuple}")
```


差別通常很明顯：含糊版長度不可控、格式每次都不一樣。

特別留意一件事：**含糊版很可能用簡體中文回答你**。Gemini 的預設語言會跟著
訓練資料的統計走，你沒指定，它就不保證。這是繁中使用者最常踩到的坑，
而修法只是在 instruction 裡加一句「一律用繁體中文」。

**實務心法**：instruction 要寫成「規格」而不是「願望」。
「回答要簡潔」是願望，「最多三點、每點不超過兩行」是規格。

## 3. `description`：寫給**其他 agent** 看的

這是最常被誤解的欄位。很多人把它當成註解隨便填，甚至留空。

但在多 agent 系統裡，`description` 是**路由的依據**——上層 agent 要決定
「這個任務該交給誰」時，它看的就是每個 sub-agent 的 `description`。

| 欄位 | 誰會讀到 | 寫法 |
|---|---|---|
| `instruction` | 這個 agent 背後的模型 | 「你要怎麼做」 |
| `description` | **父 agent 的模型** | 「我能處理什麼」 |

所以 `description` 的正確寫法是「能力宣告」，而且要**寫得能跟兄弟 agent 區分開**。

In [4]:
# ❌ 沒有區分度：父 agent 無從選擇
bad_a = LlmAgent(name="agent_a", model=get_model(),
                 description="處理使用者的問題", instruction="...")
bad_b = LlmAgent(name="agent_b", model=get_model(),
                 description="幫助使用者", instruction="...")

# ✅ 有區分度：邊界清楚
good_a = LlmAgent(name="refund_agent", model=get_model(),
                  description="處理退貨與退款：查退款進度、計算可退金額、送出退款申請。",
                  instruction="...")
good_b = LlmAgent(name="shipping_agent", model=get_model(),
                  description="處理物流與配送：查詢包裹位置、修改收件地址、重新配送。",
                  instruction="...")

for a in (bad_a, bad_b, good_a, good_b):
    print(f"{a.name:16s} → {a.description}")

agent_a          → 處理使用者的問題
agent_b          → 幫助使用者
refund_agent     → 處理退貨與退款：查退款進度、計算可退金額、送出退款申請。
shipping_agent   → 處理物流與配送：查詢包裹位置、修改收件地址、重新配送。


> 第 09 章會做一個實驗：只改 `description`、其他都不動，
> 觀察父 agent 的路由決策如何整個翻盤。

## 4. instruction 裡的動態內容

instruction 不必寫死。用 `{變數名}` 就能把 **session state** 的值注入進去。

- `{name}` — 找不到會直接報錯
- `{name?}` — 找不到就當成空字串（**教材與正式環境都建議用這個**）

In [5]:
from google.adk.runners import InMemoryRunner

personalized = LlmAgent(
    name="personal_tutor",
    model=get_model(),
    instruction=(
        "你是 {student_name?} 的私人家教，他的程度是 {level?}。\n"
        "請依照他的程度調整用詞深淺，用繁體中文回答，控制在三句話內。"
    ),
)

runner = InMemoryRunner(agent=personalized, app_name="concept_track")
sid = await new_session(runner, state={"student_name": "小明", "level": "完全新手"})

print(await ask(runner, "什麼是 API？", session_id=sid))

API 就像是餐廳的菜單，你透過菜單告訴廚房（程式）你想點什麼菜（功能），廚房就做出菜給你。它讓不同的程式可以互相溝通，就像你和廚房的對話一樣。


同一個 agent，換一組 state 就換一種說法：

In [6]:
sid2 = await new_session(runner, state={"student_name": "Sean", "level": "五年後端經驗"})
print(await ask(runner, "什麼是 API？", session_id=sid2))

API（Application Programming Interface），我們可以把它想像成餐廳裡的菜單。

菜單列出了所有你能點的菜（服務），並說明了你需要提供什麼（參數），以及你會得到什麼（回傳值）。

簡單來說，API 就是讓不同的軟體應用程式之間能夠互相溝通、交換資訊的一種約定好的介面。


### 為什麼一定要用 `{var?}`

少了問號，state 裡剛好沒這個 key 的時候會整個炸掉：

In [7]:
strict = LlmAgent(
    name="strict_bot",
    model=get_model(),
    instruction="你在服務 {student_name} 這位學生。",  # 注意：沒有問號
)
strict_runner = InMemoryRunner(agent=strict, app_name="concept_track")
sid3 = await new_session(strict_runner)  # 空 state

try:
    await ask(strict_runner, "你好", session_id=sid3)
    print("沒有報錯")
except Exception as exc:
    print(f"{type(exc).__name__}: {str(exc)[:200]}")

KeyError: 'Context variable not found: `student_name`.'


## 5. `global_instruction`：一次套用到整棵 agent 樹

多 agent 系統常有「所有 agent 都要遵守」的共同規則（語言、語氣、法遵限制）。
寫在每個 agent 的 instruction 裡會重複又難維護。

`global_instruction` 設在 **root agent** 上，整棵樹都會吃到。

In [8]:
child = LlmAgent(
    name="child_agent",
    model=get_model(),
    description="回答一般問題。",
    instruction="簡短回答使用者的問題。",
)

root = LlmAgent(
    name="root_agent",
    model=get_model(),
    global_instruction="無論如何都只能用繁體中文回答，而且結尾一定要加上「☕」。",
    instruction="你是總機，把問題交給 child_agent 處理。",
    sub_agents=[child],
)

print(await run_once(root, "Tell me a fun fact about octopuses."))

章魚有三個心臟，其中兩個負責將血液輸送到鰓部，而第三個則將血液輸送到身體的其他部位。☕


執行上面這個 cell 時，ADK 可能會多印一行提示：

> *App "..." can transfer between agents but has no `context_cache_config`.
> Every transfer swaps the system instruction and the tool set, so the request
> prefix changes and the whole prompt is re-sent uncached after each transfer.*

這是 ADK 2.x 才有的成本提醒：**每一次交棒都會讓 prompt 前綴改變，快取整個失效**。
多 agent 系統的帳單常常就是這樣爆的。解法（`context_cache_config`）在第 06 章。

## 本章重點

- **`name`** 是事件串流裡的署名，多 agent 時是唯一的辨識依據。
- **`instruction`** 給自己的模型看，要寫成「規格」不是「願望」。
- **`description`** 給**父 agent 的模型**看，是多 agent 路由的依據——
  不是註解，而且要寫得能跟兄弟 agent 區分開。
- **`{var?}`** 比 `{var}` 安全：少了問號，state 缺 key 就整個炸。
- **`global_instruction`** 設在 root，整棵樹通用。

## 動手練習

1. 把第 2 節的「精確版」再加一條規則：「如果問題跟 Python 無關，
   直接回答『這超出我的範圍』」。然後問它一個天氣問題，看規則有沒有生效。
2. 第 4 節的 state 加一個 `{tone?}` 欄位（例如 `"嚴厲"` / `"溫柔"`），
   觀察同一個問題的語氣變化。
3. 把第 5 節的 `global_instruction` 改成「只能用英文」，
   但 `child_agent` 的 instruction 寫「只能用日文」，猜猜看誰贏？跑跑看。

---
**下一站 → `02_tools.ipynb`**：讓 agent 真的能做事。